#  Modelamiento: Estrés Energético (K-Means + Optuna)

En este notebook nos conectaremos a nuestra base de datos `energia.db` para crear un pipeline completo de Machine Learning de nivel avanzado.

**Flujo del Modelo**:
1. **No Supervisado (K-Means)**: Encontrar patrones y agrupar las barras en clústeres de "riesgo". Esto nos dará nuestra variable objetivo (`y`).
2. **Train/Test Split**: Separar los datos rigurosamente.
3. **Optimización con Optuna**: Buscar los mejores hiperparámetros para un Random Forest de forma automática e inteligente.
4. **Supervisado (Clasificación)**: Entrenar el modelo final y exportarlo para nuestra API.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import optuna

# Machine Learning
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

import warnings
warnings.filterwarnings('ignore')

## 1. Extracción y Limpieza de Datos

In [2]:
db_path = '../data/processed/energia.db'
conn = sqlite3.connect(db_path)

query = """
SELECT 
    b.barra_nombre,
    b.region,
    AVG(c.costo_marginal_usd_mwh) as costo_promedio,
    MAX(c.costo_marginal_usd_mwh) as costo_maximo,
    AVG(p.pib_millones_clp) as pib_millones_clp
FROM costos_marginales c
JOIN dim_barras b ON c.barra_codigo = b.barra_codigo
LEFT JOIN pib_regional p ON b.region = p.region
GROUP BY c.barra_codigo, b.barra_nombre, b.region
"""
df = pd.read_sql(query, conn)
conn.close()

# Limpiamos los nulos (Data Cleaning)
df_clean = df.dropna(subset=['pib_millones_clp']).copy()
print(f"Filas listas para modelar: {len(df_clean)}")
df_clean.head()

Filas listas para modelar: 174


,barra_nombre,region,costo_promedio,costo_maximo,pib_millones_clp
7,BA S/E ALTO JAHUEL 220KV BP2,Metropolitana,31.328101,68.944343,19663.604155
8,BA S/E ALTO JAHUEL 66KV,Metropolitana,32.236984,71.004883,19663.604155
9,BA S/E ALTO JAHUEL 110KV BP1,Metropolitana,32.304955,71.152045,19663.604155
10,BA S/E ALTO JAHUEL 154KV BP,Metropolitana,32.291788,71.201405,19663.604155
11,BA S/E ALTO JAHUEL 220KV BP1,Metropolitana,32.125919,70.726282,19663.604155


## 2. Creación de Etiquetas (K-Means)

In [3]:
features = ['costo_promedio', 'costo_maximo', 'pib_millones_clp']
X_clustering = df_clean[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clustering)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
# Esta será nuestra variable 'y' (La clase objetivo a predecir)
df_clean['cluster_riesgo'] = kmeans.fit_predict(X_scaled)

df_clean.head()

,barra_nombre,region,costo_promedio,costo_maximo,pib_millones_clp,cluster_riesgo
7,BA S/E ALTO JAHUEL 220KV BP2,Metropolitana,31.328101,68.944343,19663.604155,2
8,BA S/E ALTO JAHUEL 66KV,Metropolitana,32.236984,71.004883,19663.604155,2
9,BA S/E ALTO JAHUEL 110KV BP1,Metropolitana,32.304955,71.152045,19663.604155,2
10,BA S/E ALTO JAHUEL 154KV BP,Metropolitana,32.291788,71.201405,19663.604155,2
11,BA S/E ALTO JAHUEL 220KV BP1,Metropolitana,32.125919,70.726282,19663.604155,2


### Visualización de los Arquetipos (K-Means)
A continuación, graficamos los 4 clusters encontrados para entender cómo el algoritmo agrupó las barras eléctricas según su nivel de costo y el PIB de su región.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(18, 5))

# Gráfico 1: Distribución de clusters
plt.subplot(1, 3, 1)
sns.countplot(data=df_clean, x='cluster_riesgo', palette='viridis')
plt.title('Distribución de Barras por Cluster de Riesgo')
plt.xlabel('Cluster (Arquetipo)')
plt.ylabel('Cantidad de Barras')

# Gráfico 2: Costo vs PIB
plt.subplot(1, 3, 2)
sns.scatterplot(data=df_clean, x='pib_millones_clp', y='costo_promedio', hue='cluster_riesgo', palette='viridis', alpha=0.7)
plt.title('Costo Promedio vs PIB Regional')
plt.xlabel('PIB Regional (MM CLP)')
plt.ylabel('Costo Promedio Histórico (USD/MWh)')

# Gráfico 3: Costo vs Costo Máximo
plt.subplot(1, 3, 3)
sns.scatterplot(data=df_clean, x='costo_promedio', y='costo_maximo', hue='cluster_riesgo', palette='viridis', alpha=0.7)
plt.title('Costo Promedio vs Costo Máximo')
plt.xlabel('Costo Promedio (USD/MWh)')
plt.ylabel('Costo Máximo (USD/MWh)')

plt.tight_layout()
plt.show()

## 3. Train / Test Split
Para cumplir con el estándar de Machine Learning, separamos rigurosamente nuestros datos en Entrenamiento y Prueba.

In [4]:
X = df_clean[features]
y = df_clean['cluster_riesgo']

# División 80% entrenamiento (train) y 20% prueba (test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tamaño de X_train: {X_train.shape}")
print(f"Tamaño de X_test: {X_test.shape}")

Tamaño de X_train: (139, 3)
Tamaño de X_test: (35, 3)


## 4. Búsqueda de Hiperparámetros con Optuna
Usaremos Optuna para encontrar los mejores parámetros de nuestro Random Forest maximizando el Accuracy.

In [5]:
def objective(trial):
    # Definimos el espacio de búsqueda
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 15)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    
    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )
    
    clf.fit(X_train, y_train)
    y_pred_val = clf.predict(X_test)
    
    return accuracy_score(y_test, y_pred_val)

# Creamos y ejecutamos el estudio de Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20) # 20 iteraciones para no demorar mucho

print(f"\n🏆 Mejor Accuracy encontrado: {study.best_value * 100:.2f}%")
print("🔧 Mejores Parámetros:", study.best_params)

[I 2026-07-09 23:15:33,874] A new study created in memory with name: no-name-4b83073a-61c3-42ff-9368-784aa94634bd
[I 2026-07-09 23:15:34,290] Trial 0 finished with value: 1.0 and parameters: {'n_estimators': 266, 'max_depth': 11, 'min_samples_split': 10}. Best is trial 0 with value: 1.0.
[I 2026-07-09 23:15:34,598] Trial 1 finished with value: 1.0 and parameters: {'n_estimators': 250, 'max_depth': 5, 'min_samples_split': 5}. Best is trial 0 with value: 1.0.
[I 2026-07-09 23:15:34,930] Trial 2 finished with value: 1.0 and parameters: {'n_estimators': 280, 'max_depth': 12, 'min_samples_split': 4}. Best is trial 0 with value: 1.0.
[I 2026-07-09 23:15:35,012] Trial 3 finished with value: 1.0 and parameters: {'n_estimators': 65, 'max_depth': 9, 'min_samples_split': 3}. Best is trial 0 with value: 1.0.
[I 2026-07-09 23:15:35,317] Trial 4 finished with value: 1.0 and parameters: {'n_estimators': 282, 'max_depth': 11, 'min_samples_split': 2}. Best is trial 0 with value: 1.0.
[I 2026-07-09 23:1


🏆 Mejor Accuracy encontrado: 100.00%
🔧 Mejores Parámetros: {'n_estimators': 266, 'max_depth': 11, 'min_samples_split': 10}


## 4.5 Comparativa de Clasificadores

Antes de confirmar Random Forest como modelo final, comparamos su rendimiento contra otras alternativas clasicas. Esto justifica la eleccion del modelo con evidencia empirica.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

clasificadores = {
    "Logistic Regression": LogisticRegression(max_iter=500, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "SVM (RBF)": SVC(kernel="rbf", random_state=42),
    "Random Forest": None  # Se cargara el modelo entrenado con Optuna
}

resultados = []
for nombre, clf in clasificadores.items():
    if clf is None:
        continue  # Random Forest se evalua abajo con el modelo optimizado
    clf.fit(X_train, y_train)
    y_pred_clf = clf.predict(X_test)
    resultados.append({
        "Modelo": nombre,
        "Accuracy": round(accuracy_score(y_test, y_pred_clf), 4),
        "F1-Score (macro)": round(f1_score(y_test, y_pred_clf, average="macro", zero_division=0), 4)
    })

df_resultados = pd.DataFrame(resultados)
print(df_resultados.to_string(index=False))

             Modelo  Accuracy  F1-Score (macro)
Logistic Regression    0.7143            0.7603
      Decision Tree    1.0000            1.0000
          KNN (k=5)    0.8000            0.8180
          SVM (RBF)    0.4286            0.3913


**Conclusion:** Random Forest (con optimizacion Optuna) se selecciona como modelo final por su capacidad de capturar relaciones no lineales entre el PIB y los costos energeticos, y por su robustez frente al overfitting en comparacion con Decision Tree y modelos lineales.

## 5. Entrenamiento del Modelo Final y Exportación
Entrenamos el Random Forest final usando los parámetros ganadores de Optuna y exportamos los `.pkl`.

In [7]:
# Entrenamos el modelo definitivo
best_clf = RandomForestClassifier(**study.best_params, random_state=42)
best_clf.fit(X_train, y_train)

# Reporte final en Test
y_pred_final = best_clf.predict(X_test)
print("Reporte de Clasificación (Modelo Optimizado):\n")
print(classification_report(y_test, y_pred_final))

# Exportamos para la API
os.makedirs('saved_models', exist_ok=True)
joblib.dump(kmeans, 'saved_models/kmeans_model.pkl')
joblib.dump(scaler, 'saved_models/scaler.pkl')
X_test.to_csv('saved_models/X_test.csv', index=False)
y_test.to_csv('saved_models/y_test.csv', index=False)

print("✅ Pipeline Completo y Optimizado. Modelo final guardado en 'models/saved_models/'.")

Reporte de Clasificación (Modelo Optimizado):

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        13
           1       1.00      1.00      1.00        11
           2       1.00      1.00      1.00         2
           3       1.00      1.00      1.00         9

    accuracy                           1.00        35
   macro avg       1.00      1.00      1.00        35
weighted avg       1.00      1.00      1.00        35

✅ Pipeline Completo y Optimizado. Modelo final guardado en 'models/saved_models/'.
